# Build a Database Advisor Agent with the DeepWiki Connector

You need a database for write-heavy local analytics. SQLite, DuckDB, and LevelDB are all strong contenders, but which one actually fits? Rather than reading documentation by hand, this notebook lets Mistral read their actual source code via the [DeepWiki](https://deepwiki.com) Connector and decide.

This notebook demonstrates the full [Mistral Connector](https://docs.mistral.ai/studio-api/connectors) lifecycle:

| Step | Operation | What happens |
|---|---|---|
| 1 | **Create** | Register a connector for each database candidate |
| 2 | **List** | Verify all three are registered |
| 3 | **Use** | Build an agent that compares them via their GitHub repos |
| 4 | **Update** | Mark the winner's connector as selected |
| 5 | **Delete** | Clean up the losing connectors |

> **API status:** This notebook uses `client.beta.connectors` and `client.beta.agents`. These are **beta** endpoints and may change.

Run cells top-to-bottom. A TypeScript version of the same agent is also available [here](./01-build-a-database-advisor-agent-typescript.md).

## Prerequisites

To complete this notebook, you will need:
- Python 3.9 or later
- A Mistral account and API key

## Environment setup

Install the Mistral Python SDK by running the cell below.

To complete this cookbook, you'll need a Mistral API key. In [Studio](https://console.mistral.ai), navigate to the [API keys section](https://console.mistral.ai/home?profile_dialog=api-keys), choose **Private and shared connectors** for **Connector access scope** and create a new API key. 

Set it before running the client cell using one of these options:

**Option 1 — environment variable** (recommended for local use):

```
MISTRAL_API_KEY=your-mistral-api-key
```

**Option 2 — enter it when prompted**: if `MISTRAL_API_KEY` is not already set in your environment, the next code cell will display a secure input field where you can paste your key directly.

In [ ]:
%pip install mistralai --quiet

Import the SDK and create the client. If `MISTRAL_API_KEY` is not set as an environment variable, a secure input prompt will appear.

In [13]:
import getpass
import os
import re

from mistralai.client import Mistral

if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Mistral API key: ")

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

## Step 1 — Create one connector per candidate

Each connector points at the [DeepWiki](https://deepwiki.com) MCP server, which lets Mistral read and reason about any public GitHub repository. We create three named Connectors (one per candidate) so each one acts as a named slot the agent can query independently.

All three point at the same MCP server URL; the connector names are what distinguish them when the agent decides which tools to call.

In [16]:
DEEPWIKI_URL = "https://mcp.deepwiki.com/mcp"

candidates = [
    {"name": "showdown_sqlite",  "description": "DeepWiki connector — sqlite/sqlite"},
    {"name": "showdown_duckdb",  "description": "DeepWiki connector — duckdb/duckdb"},
    {"name": "showdown_leveldb", "description": "DeepWiki connector — google/leveldb"},
]

connectors = {}
for c in candidates:
    connector = await client.beta.connectors.create_async(
        name=c["name"],
        description=c["description"],
        server=DEEPWIKI_URL,
        visibility="private",
    )
    connectors[c["name"]] = connector
    print(f"Created: {connector.name}  (id={connector.id})")

Created: showdown_sqlite  (id=019f8ed1-7c70-7630-b6f0-80478c2e3bb0)
Created: showdown_duckdb  (id=019f8ed1-854e-718a-85db-1e0dd2638a69)
Created: showdown_leveldb  (id=019f8ed1-8c3f-764e-b38b-d087ec65044e)


## Step 2 — List to verify

Confirm all three connectors are registered before proceeding.

View your registered Connectors in [Studio](https://console.mistral.ai/build/connectors).

In [17]:
page = await client.beta.connectors.list_async(page_size=50)
showdown = [c for c in page.items if c.name.startswith("showdown_")]

print(f"{len(showdown)} showdown connectors registered:\n")
for c in showdown:
    print(f"  {c.name:<22}  {c.description}")

3 showdown connectors registered:

  showdown_sqlite         DeepWiki connector — sqlite/sqlite
  showdown_duckdb         DeepWiki connector — duckdb/duckdb
  showdown_leveldb        DeepWiki connector — google/leveldb


## Step 3 — Build the comparison agent

We create a Mistral agent with all three connectors attached. The agent's instructions do two things:

1. **Direct the agent to use the connectors** — it must ask each DeepWiki connector a natural-language question about the repository before forming an opinion. Asking questions rather than reading raw source files keeps the responses concise enough to fit in the context window.

2. **Require JSON output** — the agent must return a single JSON object matching the schema defined below. The schema is defined in code so it serves as a precise, readable contract for the expected structure. Note that the conversations API does not support `response_format` alongside `agent_id`, so JSON conformance is enforced via the agent's instructions rather than at the API level.

### Response schema

We define the expected JSON structure before creating the agent. The schema has four required fields:

- **`queries`** — one entry per database, capturing the question sent to each connector and the summary returned. This makes the connector calls visible in the output rather than hidden inside the model's reasoning.
- **`comparison`** — a fixed set of evaluation dimensions (`storage_model`, `acid_guarantees`, `query_capabilities`, `write_throughput`, `python_api`), each as a brief comparative string.
- **`reasoning`** — a single paragraph explaining the final choice.
- **`recommendation`** — one of the three connector names, constrained to an `enum` so the value is always a valid key in `connectors`.

Defining the schema in code (rather than describing it in the instruction string) means it serves as the single source of truth: the same object is referenced in the agent's instructions and passed to the API as a `json_schema` response format, so the structure is enforced at the API level rather than relying on the model to follow prose directions.

In [ ]:
RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "queries": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "connector":  {"type": "string"},
                    "question":   {"type": "string"},
                    "summary":    {"type": "string"},
                },
                "required": ["connector", "question", "summary"],
            },
        },
        "comparison": {
            "type": "object",
            "properties": {
                "storage_model":      {"type": "string"},
                "acid_guarantees":    {"type": "string"},
                "query_capabilities": {"type": "string"},
                "write_throughput":   {"type": "string"},
                "python_api":         {"type": "string"},
            },
            "required": ["storage_model", "acid_guarantees", "query_capabilities", "write_throughput", "python_api"],
        },
        "reasoning":      {"type": "string"},
        "recommendation": {"type": "string", "enum": ["showdown_sqlite", "showdown_duckdb", "showdown_leveldb"]},
    },
    "required": ["queries", "comparison", "reasoning", "recommendation"],
}

Create the agent with the three connectors attached. The `completion_args` field passes `RESPONSE_SCHEMA` to the API as a `json_schema` response format, so the model is constrained to return valid JSON matching the schema on every run.

In [28]:
from mistralai.client.models import CompletionArgs, JSONSchema, ResponseFormat

agent = await client.beta.agents.create_async(
    name="Database Showdown Judge",
    description="Compares database candidates using their source code via DeepWiki.",
    model="mistral-medium-latest",
    instructions=(
        "You are a database selection expert. "
        "When given a comparison task, call each DeepWiki connector once with a focused "
        "natural-language question about the repository — do NOT read raw source files."
    ),
    completion_args=CompletionArgs(
        response_format=ResponseFormat(
            type="json_schema",
            json_schema=JSONSchema(
                name="comparison_result",
                schema_definition=RESPONSE_SCHEMA,
                strict=True,
            ),
        ),
    ),
    tools=[
        {"type": "connector", "connector_id": connectors["showdown_sqlite"].id},
        {"type": "connector", "connector_id": connectors["showdown_duckdb"].id},
        {"type": "connector", "connector_id": connectors["showdown_leveldb"].id},
    ],
)
print(f"Agent ready: {agent.name}  (id={agent.id})")

Agent ready: Database Showdown Judge  (id=ag_019f8f1f05bb72a0894d93e8803068f5)


## Step 4 — Run the comparison

Send the comparison question to the agent. Before replying, the agent will call each DeepWiki connector once with a natural-language question — this may take a minute.

The cell does two things as it processes the response:

- **Logs tool calls as they happen** — any output that isn't the final message (connector calls, internal steps) is printed with its type and arguments, so you can confirm the connectors are being invoked.
- **Parses and displays the JSON result** — once the agent replies, the JSON is parsed and printed in a readable format: the question asked of each connector and the summary returned, the dimension-by-dimension comparison, the reasoning, and the final recommendation.

In [29]:
import json

response = await client.beta.conversations.start_async(
    agent_id=agent.id,
    inputs=[
        {
            "role": "user",
            "content": (
                "Compare sqlite/sqlite, duckdb/duckdb, and google/leveldb for a write-heavy "
                "local analytics workload. Evaluate storage model, ACID guarantees, query "
                "capabilities, write throughput, and Python API simplicity. Recommend one."
            ),
        }
    ],
)

# Collect the agent's full reply and surface tool call activity
raw_text = ""
for output in response.outputs:
    if output.type == "message.output":
        content = output.content
        if isinstance(content, str):
            raw_text += content
        else:
            raw_text += "".join(
                chunk.text if hasattr(chunk, "text") else str(chunk)
                for chunk in content
            )
    else:
        name = getattr(output, "name", "") or getattr(output, "tool_name", "")
        args = getattr(output, "arguments", None)
        detail = f" — {name}" if name else ""
        if args:
            try:
                parsed = json.loads(args) if isinstance(args, str) else args
                detail += f"\n    {json.dumps(parsed, indent=2)}"
            except (ValueError, TypeError):
                detail += f"\n    {args}"
        print(f"[{output.type}]{detail}")

# The agent enforces json_schema via completion_args, so the response is
# guaranteed to be valid JSON matching RESPONSE_SCHEMA.
result = json.loads(raw_text)

print("\n--- Connector queries ---")
for q in result.get("queries", []):
    print(f"\n  [{q['connector']}]")
    print(f"  Q: {q['question']}")
    print(f"  A: {q['summary']}")

print("\n--- Comparison ---")
for key, val in result.get("comparison", {}).items():
    print(f"  {key}: {val}")

print(f"\n--- Reasoning ---\n  {result.get('reasoning', '')}")
print(f"\n--- Recommendation ---\n  {result.get('recommendation', '')}")


--- Connector queries ---

  [showdown_sqlite]
  Q: What are SQLite's storage model, ACID guarantees, query capabilities, write throughput characteristics, and Python API simplicity for a write-heavy local analytics workload?
  A: Investigate SQLite's storage model, ACID compliance, query support, write performance, and Python API for local analytics.

  [showdown_duckdb]
  Q: What are DuckDB's storage model, ACID guarantees, query capabilities, write throughput characteristics, and Python API simplicity for a write-heavy local analytics workload?
  A: Investigate DuckDB's storage model, ACID compliance, query support, write performance, and Python API for local analytics.

  [showdown_leveldb]
  Q: What are LevelDB's storage model, ACID guarantees, query capabilities, write throughput characteristics, and Python API simplicity for a write-heavy local analytics workload?
  A: Investigate LevelDB's storage model, ACID compliance, query support, write performance, and Python API for loc

Because the agent returned structured JSON, extracting the winner is a direct key lookup — no regex required. We also validate that the recommendation is one of the known connector names before proceeding.

In [ ]:
winner_name = result.get("recommendation", "").strip()
if winner_name not in connectors:
    raise ValueError(f"Unexpected recommendation {winner_name!r} — expected one of {list(connectors)}")

loser_names = [name for name in connectors if name != winner_name]

print(f"Winner: {winner_name}")
print(f"Losers: {', '.join(loser_names)}")

## Step 5 — Promote the winner, retire the rest

Update the winning connector's description to mark it as selected, then delete the losing connectors. This completes the full lifecycle: create → list → use → update → delete.

In [ ]:
# Mark the winner
winner_connector = connectors[winner_name]
updated = await client.beta.connectors.update_async(
    connector_id=winner_connector.id,
    description=f"[SELECTED] {winner_connector.description}",
)
print(f"Updated:  {updated.name}  —  {updated.description}")

# Delete the losers
for name in loser_names:
    delete_result = await client.beta.connectors.delete_async(connector_id=connectors[name].id)
    print(f"Deleted:  {name}  —  {delete_result.message}")

Confirm the winner connector is still registered with its updated description.

In [9]:
# Confirm the winner is still there with its updated description
winner = await client.beta.connectors.get_async(connector_id_or_name=winner_name)
print("Winner confirmed:")
print(f"  Name:        {winner.name}")
print(f"  Description: {winner.description}")
print(f"  ID:          {winner.id}")

Winner confirmed:
  Name:        showdown_duckdb
  Description: [SELECTED] DeepWiki connector — duckdb/duckdb
  ID:          019f8eca-b600-729b-af89-4b494aeb0573


## Cleanup

Delete the agent when you're done. Uncomment the last two lines to also remove the winning Connector.

In [10]:
await client.beta.agents.delete_async(agent_id=agent.id)
print(f"Agent deleted: {agent.id}")

# Uncomment to also remove the winning Connector:
# result = await client.beta.connectors.delete_async(connector_id=winner_connector.id)
# print(f"Connector deleted: {winner_name}")

Agent deleted: ag_019f8ecabf2e7786833c5be616c80a13


## Summary

This notebook demonstrated the full Mistral Connector lifecycle — create, list, use, update, and delete — using the DeepWiki Connector to let the model read actual GitHub repository source code and produce a data-driven database recommendation.

**What you built:**
- Three named Connectors pointing at the DeepWiki MCP server
- An agent (Database Showdown Judge) with all three Connectors attached
- A conversation that produced a structured recommendation, updated the winner's Connector, and cleaned up the rest

**Mistral features used:**
- Connectors (beta)
- Agents API (beta)
- Conversations API (beta)

**Other services:**
- [DeepWiki](https://deepwiki.com) — MCP server for reading public GitHub repositories

View your Connectors in [Studio](https://console.mistral.ai/build/connectors).